# Notebook 3 of 3: Cell type mapping with cell2location (GPU)

**What this notebook does.** It assigns a cell type to every cell in your spatial sample by comparing it to an
annotated single cell reference (scRNA-seq that already has cell type labels). This is the automatic, tissue
neutral alternative to naming clusters by hand in Notebook 2. cell2location first learns an expression
signature for each cell type from the reference, then estimates how much of each cell type sits at every
location in your spatial data.

**Where to run this.** cell2location needs an NVIDIA GPU to run in reasonable time. Run it on the same GPU
machine as Notebook 1, or on Google Colab with a GPU runtime (Runtime, then Change runtime type, then GPU). It
will run on a laptop CPU but can take many hours, so CPU is only for a tiny test.

**What you need.**
- Your spatial sample as an `.h5ad`. Either the raw file from Notebook 1 (`SAMPLE_cpsam_proseg_raw.h5ad`) or the
  annotated file from Notebook 2 (`SAMPLE_annotated.h5ad`) works.
- A reference `.h5ad` for the **same tissue**, with raw counts and a column of cell type labels. You bring your
  own reference that matches your sample. For keloid skin, use a keloid or skin scRNA-seq reference. For mouse
  lung, use a mouse lung (BPD) reference. cell2location can only find cell types that exist in the reference, so
  the reference has to match the tissue.

**What it produces.**
- `SAMPLE_cell2location.h5ad`, your spatial object with a cell type abundance per cell and a `cell2loc_type`
  label (the most likely type for each cell).
- A spatial map coloured by `cell2loc_type`.

**How to run.** Click a cell, press **Shift+Enter**, top to bottom. Edit only the lines marked `# EDIT`.

## Set up once

cell2location brings its own heavy tools (scvi-tools and PyTorch). Install it in its own environment on the GPU
machine:

```bash
conda create -n cell2loc python=3.10 -y
conda activate cell2loc
pip install cell2location jupyterlab ipykernel
python -m ipykernel install --user --name cell2loc --display-name "Python (cell2loc)"
jupyter lab
```

On Google Colab instead: start a new notebook, set Runtime, then Change runtime type, then GPU, and run this in
the first cell:

```python
!pip install cell2location
```

Then upload your two `.h5ad` files (the spatial one and the reference) using the file panel on the left.

## Step 1. Settings

Set the sample name, the two input paths, the reference cell type column, and the output folder, then run. It
prints the shapes so you can confirm both files opened, and lists the reference labels so you can confirm the
column is the right one.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import scanpy as sc, anndata as ad
import scipy.sparse as sp
from pathlib import Path
import cell2location

SAMPLE     = "Y40172EA"                                              # EDIT: sample name
SPATIAL_IN = Path("/path/to/Y40172EA_cpsam_proseg_raw.h5ad")        # EDIT: your spatial .h5ad (Notebook 1 or 2 output)
REF_IN     = Path("/path/to/your_reference.h5ad")                   # EDIT: your annotated reference .h5ad (same tissue)
REF_LABEL_KEY = "cell_type"      # EDIT: the reference column that holds the cell type labels
REF_BATCH_KEY = None             # EDIT: reference batch/sample column if it has one (e.g. "donor"), else None
OUT        = Path("/path/to/cell2loc_out"); OUT.mkdir(parents=True, exist_ok=True)   # EDIT: output folder

adata = sc.read_h5ad(SPATIAL_IN)
ref   = sc.read_h5ad(REF_IN)
print("spatial  :", adata.shape)
print("reference:", ref.shape)
print("\nreference cell type labels in", REF_LABEL_KEY, ":")
print(ref.obs[REF_LABEL_KEY].value_counts().to_string())

## Step 2. Make sure both objects hold raw counts

cell2location needs raw integer counts, not normalised values. Notebook 1 output already has raw counts in `X`.
Notebook 2 output keeps them in the `counts` layer, so this restores them. It does the same for the reference. If
your reference already has raw counts in `X`, this changes nothing. The two lines should print `True`.

In [ ]:
# spatial: use the raw counts
if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()
adata.var_names_make_unique()

# reference: use the raw counts
if "counts" in ref.layers:
    ref.X = ref.layers["counts"].copy()
ref.var_names_make_unique()

def looks_integer(X):
    x = X[:50].toarray() if sp.issparse(X) else np.asarray(X[:50])
    return bool(np.allclose(x, np.round(x)))

print("spatial counts look integer  :", looks_integer(adata.X), "| max:", adata.X.max())
print("reference counts look integer:", looks_integer(ref.X), "| max:", ref.X.max())

# drop reference cells that have no cell type label
ref = ref[ref.obs[REF_LABEL_KEY].notna()].copy()
print("reference cells with a label :", ref.n_obs)

## Step 3. Learn a signature for each cell type from the reference

This filters the reference down to informative genes, then trains a model that estimates the average expression
of every gene in every cell type. This is the slow reference step, a few minutes on a GPU. The training plot
should slope down and flatten out.

In [ ]:
from cell2location.utils.filtering import filter_genes
from cell2location.models import RegressionModel

# keep genes that are informative across cell types
selected = filter_genes(ref, cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)
ref = ref[:, selected].copy()

RegressionModel.setup_anndata(adata=ref, labels_key=REF_LABEL_KEY, batch_key=REF_BATCH_KEY)
reg = RegressionModel(ref)
reg.train(max_epochs=250)
reg.plot_history(); plt.show()

ref = reg.export_posterior(ref, sample_kwargs={"num_samples": 1000, "batch_size": 2500})

# pull out the per cell type signature table (genes x cell types)
if "means_per_cluster_mu_fg" in ref.varm.keys():
    inf_aver = ref.varm["means_per_cluster_mu_fg"][[f"means_per_cluster_mu_fg_{i}"
                                                    for i in ref.uns["mod"]["factor_names"]]].copy()
else:
    inf_aver = ref.var[[f"means_per_cluster_mu_fg_{i}" for i in ref.uns["mod"]["factor_names"]]].copy()
inf_aver.columns = ref.uns["mod"]["factor_names"]
print("signature table:", inf_aver.shape, "(genes x cell types)")

## Step 4. Map the cell types onto your spatial data

This matches genes between the reference signatures and your spatial data, then estimates the amount of each cell
type at every cell. Two knobs:

- `N_CELLS_PER_LOCATION`. For Proseg data each location is a single cell, so **1** is right. For binned or spot
  data (like Visium), set it to the average number of cells per spot.
- `DETECTION_ALPHA`. Leave at 20. Lower it (towards 1) only if your data has strong technical differences
  between regions.

This is the long step. On a GPU it takes several minutes to tens of minutes depending on how many cells you have.

In [ ]:
N_CELLS_PER_LOCATION = 1     # EDIT: 1 for single cell (Proseg) data; average cells per spot for binned/spot data
DETECTION_ALPHA      = 20    # EDIT: leave at 20 in most cases

# keep only the genes shared by the signatures and the spatial data
shared = np.intersect1d(adata.var_names, inf_aver.index)
print("shared genes:", len(shared))
adata = adata[:, shared].copy()
inf_aver = inf_aver.loc[shared, :].copy()

cell2location.models.Cell2location.setup_anndata(adata=adata)
mod = cell2location.models.Cell2location(
    adata, cell_state_df=inf_aver,
    N_cells_per_location=N_CELLS_PER_LOCATION,
    detection_alpha=DETECTION_ALPHA,
)
mod.train(max_epochs=30000, batch_size=None, train_size=1)
mod.plot_history(1000); plt.show()

adata = mod.export_posterior(adata, sample_kwargs={"num_samples": 1000, "batch_size": adata.n_obs})

## Step 5. Give each cell its most likely type, and save

The estimated abundances (the 5 percent quantile, a conservative estimate) go into the cell table, and each cell
is labelled with the cell type of highest abundance. The result is saved as an `.h5ad`.

In [ ]:
q05 = adata.obsm["q05_cell_abundance_w_sf"]
q05 = q05.values if hasattr(q05, "values") else np.asarray(q05)
types = list(adata.uns["mod"]["factor_names"])
abund = pd.DataFrame(q05, index=adata.obs_names, columns=types)

for t in types:
    adata.obs[t] = abund[t].values                 # one abundance column per cell type
adata.obs["cell2loc_type"] = abund.idxmax(axis=1).values   # the winning type per cell
print(adata.obs["cell2loc_type"].value_counts().to_string())

FINAL = OUT / f"{SAMPLE}_cell2location.h5ad"
adata.write_h5ad(FINAL, compression="gzip")
print("\nsaved:", FINAL, "| size MB:", round(FINAL.stat().st_size / 1e6, 1))

## Step 6. Map of the cell types

A spatial map coloured by `cell2loc_type`, saved to the output folder.

In [ ]:
xy = adata.obsm["spatial"]
cats = adata.obs["cell2loc_type"].astype("category").cat.categories
cmap = plt.get_cmap("tab20").colors
fig, axp = plt.subplots(figsize=(11, 10))
for i, t in enumerate(cats):
    m = (adata.obs["cell2loc_type"] == t).values
    axp.scatter(xy[m, 0], xy[m, 1], s=3, c=[cmap[i % 20]], label=f"{t} ({m.sum():,})", linewidths=0)
axp.invert_yaxis(); axp.set_aspect("equal"); axp.axis("off")
axp.set_title(f"{SAMPLE}: cell2location cell types")
axp.legend(markerscale=4, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(OUT / f"{SAMPLE}_cell2location_map.png", dpi=200, bbox_inches="tight")
plt.show()

## Notes and troubleshooting

- **Where to run.** If `mod.train` reports it is using CPU, cell2location did not find a GPU. On the GPU machine,
  check the `cell2loc` environment sees the card. On Colab, set Runtime, then Change runtime type, then GPU,
  restart, and rerun.
- **Version differences.** cell2location changes quickly. If `train` complains about a `use_gpu` or `accelerator`
  argument, remove it and let it auto detect, or match the argument to your installed version from the
  cell2location docs.
- **Speed.** `max_epochs=30000` is the standard for the mapping step. To test the whole notebook quickly first,
  lower it (for example 5000), confirm it runs end to end, then raise it back for the final result.
- **Types come only from the reference.** cell2location can only assign types that exist in your reference
  labels. If a type is missing or a call looks wrong, check the reference actually matches your tissue.
- **Using the result.** Each cell type now has an abundance column in `adata.obs`, and `cell2loc_type` is the
  winning type per cell. You can carry `cell2loc_type` back into Notebook 2 as an independent check on the
  manual annotation.